In [1]:
!pip install -q sentence-transformers faiss-cpu pandas scikit-learn

In [2]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast

texts = [
    "I was charged twice for the same credit card payment",
    "The bank took money from my account two times for one bill",
    "Debt collector keeps calling me at work",
    "My mortgage payment was not applied correctly",
]

emb = model.encode(texts, convert_to_tensor=True)
print(util.cos_sim(emb[0], emb))  # how similar is sentence 1 to each one?$


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tensor([[1.0000, 0.6457, 0.3242, 0.4314]])


In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/aminbadh/cfpb-complaint/main/data/processed/complaints_sample.csv"
df = pd.read_csv(url)
print(df.shape)
print(df.columns.tolist())
df.head(3)

(50000, 18)
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2026-02-24,Debt collection,I do not know,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,NaN,NaN,"Affirm Holdings, Inc",CA,91202,NaN,NaN,Web,2026-02-24,Closed with explanation,Yes,NaN,19737724
1,2025-11-05,Prepaid card,General-purpose prepaid card,Problem getting a card or closing an account,Trouble closing card,On XXXX I received a prepaid debit card from N...,Company has responded to the consumer and the ...,Netspend Corporation,NC,28078,NaN,Consent provided,Web,2025-11-05,Closed with explanation,Yes,NaN,17039807
2,2023-08-01,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,77386,NaN,Other,Web,2023-08-01,Closed with non-monetary relief,Yes,NaN,7335287


In [4]:
import numpy as np, faiss

NARR = "Consumer complaint narrative"
df = df.dropna(subset=[NARR]).reset_index(drop=True)
df = df.sample(n=min(5000, len(df)), random_state=42).reset_index(drop=True)

emb = model.encode(df[NARR].tolist(), batch_size=64,
                   show_progress_bar=True, normalize_embeddings=True)
emb = np.asarray(emb, dtype="float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print(index.ntotal, "complaints indexed")
np.save("emb.npy", emb)
df.to_csv("df_5000.csv", index=False)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

5000 complaints indexed


In [5]:
def search(query, k=5):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    for s, i in zip(scores[0], ids[0]):
        print(f"{s:.3f} | {df.loc[i, 'Issue']} | {df.loc[i, NARR][:200]}...\n")

search("I was charged twice for the same payment")

0.726 | Unexpected or other fees | XXXX  billed me twice for {$1.00} when my balance was {$1.00} - I submitted both transactions but clearly only one should have gone through....

0.624 | Improper use of your report | I have confirmed that my payment was made always made within the agreed payment terms outlined in our contract. I have attached copies of the payment confirmation and transaction details for your refe...

0.606 | Problem with a purchase shown on your statement | I bought two items from amazon. I returned both of them and amazon did not give me a credit. Disputed with chase and they gave me a credit for only one and the other they said the merchant didnt reciv...

0.596 | Attempts to collect debt not owed | On XX/XX/XXXX, i went to the dentist to get all new XXXX. i had insurance that paid part of my XXXX XXXX. On XX/XX/XXXX, Imade my first payment of {$150.00} online with XXXX XXXX.My first balance was ...

0.595 | Trouble using your card | I paid my XXXX XXXX credit card

In [6]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

labels = df["Issue"].values
counts = df["Issue"].map(df["Issue"].value_counts())
valid = np.where(counts > 1)[0]               # queries that have at least one same-Issue neighbor
rng = np.random.default_rng(42)
q_idx = rng.choice(valid, size=500, replace=False)

def evaluate(S, ks=(1, 5, 10)):
    hit = {k: [] for k in ks}; prec = {k: [] for k in ks}; rr = []
    for row, qi in zip(S, q_idx):
        row = row.copy(); row[qi] = -1e9          # exclude the query itself
        order = np.argsort(-row)[:max(ks)]
        rel = labels[order] == labels[qi]
        for k in ks:
            hit[k].append(rel[:k].any()); prec[k].append(rel[:k].mean())
        rr.append(1 / (np.argmax(rel) + 1) if rel.any() else 0)
    out = {f"Hit@{k}": np.mean(hit[k]) for k in ks}
    out.update({f"P@{k}": np.mean(prec[k]) for k in ks})
    out["MRR"] = np.mean(rr)
    return out

# Embedding search (you already have emb)
S_emb = emb[q_idx] @ emb.T

# TF-IDF baseline
tfidf = TfidfVectorizer(stop_words="english", max_features=50000, sublinear_tf=True)
X = tfidf.fit_transform(df[NARR])
S_tfidf = linear_kernel(X[q_idx], X)

results = pd.DataFrame({"Embeddings (MiniLM)": evaluate(S_emb),
                        "TF-IDF baseline": evaluate(S_tfidf)}).round(3)
results

,Embeddings (MiniLM),TF-IDF baseline
Hit@1,0.460,0.470
Hit@5,0.708,0.714
Hit@10,0.780,0.798
P@1,0.460,0.470
P@5,0.410,0.408
P@10,0.382,0.380
MRR,0.563,0.571


In [7]:
def zrow(S):
    return (S - S.mean(1, keepdims=True)) / S.std(1, keepdims=True)

rows = {"Embeddings": evaluate(S_emb), "TF-IDF": evaluate(S_tfidf)}
for a in [0.3, 0.5, 0.7]:
    S_h = a * zrow(S_emb) + (1 - a) * zrow(S_tfidf)
    rows[f"Hybrid (a={a})"] = evaluate(S_h)

pd.DataFrame(rows).round(3)

,Embeddings,TF-IDF,Hybrid (a=0.3),Hybrid (a=0.5),Hybrid (a=0.7)
Hit@1,0.460,0.470,0.476,0.478,0.498
Hit@5,0.708,0.714,0.734,0.734,0.748
Hit@10,0.780,0.798,0.814,0.816,0.808
P@1,0.460,0.470,0.476,0.478,0.498
P@5,0.410,0.408,0.418,0.421,0.427
P@10,0.382,0.380,0.389,0.395,0.397
MRR,0.563,0.571,0.583,0.585,0.602


In [8]:
def rr_vec(S):
    out = []
    for row, qi in zip(S, q_idx):
        row = row.copy(); row[qi] = -1e9
        order = np.argsort(-row)[:10]
        rel = labels[order] == labels[qi]
        out.append(1 / (np.argmax(rel) + 1) if rel.any() else 0)
    return np.array(out)

grid = [round(x, 1) for x in np.arange(0, 1.01, 0.1)]
S_h = lambda a: a * zrow(S_emb) + (1 - a) * zrow(S_tfidf)
rr = {a: rr_vec(S_h(a)) for a in grid}      # a=0 is TF-IDF only, a=1 is embeddings only

rng2 = np.random.default_rng(0)
perm = rng2.permutation(len(q_idx)); A, B = perm[:250], perm[250:]
best_a = max(grid, key=lambda a: rr[a][A].mean())   # tune on half A

print("best a (tuned on half A):", best_a)
print("MRR on held-out half B ->",
      "TF-IDF:", rr[0.0][B].mean().round(3),
      "| Embeddings:", rr[1.0][B].mean().round(3),
      "| Hybrid:", rr[best_a][B].mean().round(3))

diff = rr[best_a][B] - rr[0.0][B]
boots = [rng2.choice(diff, len(diff)).mean() for _ in range(5000)]
print("MRR gain vs TF-IDF, 95% CI:", np.percentile(boots, [2.5, 97.5]).round(3))


best a (tuned on half A): 0.7
MRR on held-out half B -> TF-IDF: 0.573 | Embeddings: 0.586 | Hybrid: 0.604
MRR gain vs TF-IDF, 95% CI: [-0.     0.061]


In [9]:
# Re-run with many more test queries
q_idx = rng.choice(valid, size=min(3000, len(valid)), replace=False)

S_emb = emb[q_idx] @ emb.T
S_tfidf = linear_kernel(X[q_idx], X)

rr = {a: rr_vec(S_h(a)) for a in grid}

rng2 = np.random.default_rng(0)
perm = rng2.permutation(len(q_idx)); h = len(perm) // 2
A, B = perm[:h], perm[h:]
best_a = max(grid, key=lambda a: rr[a][A].mean())

print("best a:", best_a, "| n_test:", len(B))
print("MRR on held-out half ->",
      "TF-IDF:", rr[0.0][B].mean().round(3),
      "| Embeddings:", rr[1.0][B].mean().round(3),
      "| Hybrid:", rr[best_a][B].mean().round(3))

diff = rr[best_a][B] - rr[0.0][B]
boots = [rng2.choice(diff, len(diff)).mean() for _ in range(5000)]
print("MRR gain vs TF-IDF, 95% CI:", np.percentile(boots, [2.5, 97.5]).round(3))

best a: 0.7 | n_test: 1500
MRR on held-out half -> TF-IDF: 0.556 | Embeddings: 0.586 | Hybrid: 0.586
MRR gain vs TF-IDF, 95% CI: [0.017 0.044]


In [10]:
def ci(x, y):
    d = rr[x][B] - rr[y][B]
    b = [rng2.choice(d, len(d)).mean() for _ in range(5000)]
    return d.mean().round(3), np.percentile(b, [2.5, 97.5]).round(3)

print("Embeddings vs TF-IDF:", ci(1.0, 0.0))
print("Hybrid vs Embeddings:", ci(best_a, 1.0))


Embeddings vs TF-IDF: (np.float64(0.031), array([0.013, 0.048]))
Hybrid vs Embeddings: (np.float64(-0.0), array([-0.015,  0.014]))
